# Simple: Correlation Analyzer with PAMOLA.CORE

**Goal**: Learn correlation analysis basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Analyze correlations between fields using execute()
- Compare different correlation methods (Pearson, Cramer's V, etc.)
- Understand correlation strength and statistical significance

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 01_correlation_analyzer_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.correlation import (
        CorrelationOperation,
        CorrelationMatrixOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from examples/data_examples/sample.csv
- Auto-creates sample data if file doesn't exist
- Review dataset structure and statistics

**What you'll see:**
- File path confirmation or creation message
- Dataset summary (200 records, 10 columns)
- First 5 records preview
- Column statistics (types, unique values, ranges)
- Mix of numeric fields (advertising_spend, sales_revenue) and categorical fields (region, sales_channel)

**Note:** Sample includes correlated data for demonstration (e.g., advertising → sales)

In [ ]:
# Define path to sample data (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

# Check if file exists
if not data_path.exists():
    print(f"⚠️  File not found: {data_path}")
    print("Creating sample data...\n")
    
    # Create directory
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample sales performance data
    np.random.seed(42)
    n_samples = 200
    
    # Generate correlated numeric data
    advertising_spend = np.random.uniform(1000, 10000, n_samples)
    sales_revenue = advertising_spend * np.random.uniform(2, 5, n_samples) + np.random.normal(0, 1000, n_samples)
    
    customer_satisfaction = np.random.uniform(1, 5, n_samples)
    repeat_purchases = (customer_satisfaction * 20) + np.random.normal(0, 10, n_samples)
    repeat_purchases = np.clip(repeat_purchases, 0, 100)
    
    # Generate categorical data
    regions = np.random.choice(['North', 'South', 'East', 'West', 'Central'], n_samples)
    product_categories = np.random.choice(['Electronics', 'Clothing', 'Books', 'Home', 'Sports'], n_samples)
    
    # Sales channels with different performance
    sales_channels = np.random.choice(['Online', 'In-Store', 'Phone'], n_samples, p=[0.5, 0.3, 0.2])
    
    # Season affects sales
    seasons = np.random.choice(['Spring', 'Summer', 'Fall', 'Winter'], n_samples)
    
    sample_data = pd.DataFrame({
        'transaction_id': [f'T{i:04d}' for i in range(1, n_samples + 1)],
        'advertising_spend': advertising_spend.round(2),
        'sales_revenue': sales_revenue.round(2),
        'customer_satisfaction': customer_satisfaction.round(2),
        'repeat_purchases_pct': repeat_purchases.round(2),
        'region': regions,
        'product_category': product_categories,
        'sales_channel': sales_channels,
        'season': seasons,
        'num_employees': np.random.randint(5, 50, n_samples)
    })
    
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created at: {data_path}\n")

# Load data using pandas
df = read_csv(data_path)

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:25s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_numeric_dtype(df[col]):
        print(f"  {col:25s} ({dtype_str:10s}): min={df[col].min():.2f}, max={df[col].max():.2f}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field1 and field2**: Edit `create_config_kwargs()` function
   - Change `"field1": "years_of_experience"` to YOUR first column
   - Change `"field2": "current_salary_cad"` to YOUR second column
   - Available fields: advertising_spend, sales_revenue, customer_satisfaction, repeat_purchases_pct, region, product_category, sales_channel, season, num_employees
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ Both fields exist with type and range/unique values)
- Task directory created (✓)
- TaskReporter initialized (✓)
- Config kwargs confirmed (✓)
- DataSource created (✓)
- Progress tracker ready (✓)

**Note:** Default fields don't exist in sample data - you MUST customize both field1 and field2!

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field1": "advertising_spend",  # First field
        "field2": "sales_revenue"        # Second field
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field1": "years_of_experience",   # First field - CUSTOMIZE THIS!
        "field2": "current_salary_cad"         # Second field - CUSTOMIZE THIS!
    }

# Get config
kwargs = create_config_kwargs()
field1 = kwargs["field1"]
field2 = kwargs["field2"]

# Validate fields exist in dataset
print(f"\n🔍 Validating field configuration...\n")
for field in [field1, field2]:
    if field not in df.columns:
        raise ValueError(
            f"❌ Field '{field}' not found in dataset!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update fields in create_config_kwargs()"
        )

print(f"✓ Fields to analyze:")
print(f"  Field 1: '{field1}'")
print(f"    Type: {df[field1].dtype}")
if pd.api.types.is_numeric_dtype(df[field1]):
    print(f"    Range: [{df[field1].min():.2f}, {df[field1].max():.2f}]")
else:
    print(f"    Unique values: {df[field1].nunique()}")

print(f"\n  Field 2: '{field2}'")
print(f"    Type: {df[field2].dtype}")
if pd.api.types.is_numeric_dtype(df[field2]):
    print(f"    Range: [{df[field2].min():.2f}, {df[field2].max():.2f}]")
else:
    print(f"    Unique values: {df[field2].nunique()}")

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="correlation_001",
    task_type="correlation_analysis",
    description=f"Correlation analysis between '{field1}' and '{field2}'.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config kwargs (without field names for execute)
execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Analyzing correlation between {field1} and {field2}",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review operation configuration
- Run to execute correlation analysis
- Monitor progress bar (6 processing steps)

**What you'll see:**
- Configuration summary (fields, method, null handling confirmed)
- Progress bar: validate → analyze → calculate → visualize → save
- Completion message with status and artifact count (2-4 files expected)
- Status = "success" (verify no errors)

**Key parameters:**
- `field1`, `field2`: Field pair to correlate (from config)
- `method=None`: Auto-detects method (Pearson, Cramer's V, etc.)
- `null_handling='drop'`: How to handle missing values
- `generate_visualization=True`: Create scatter/box/heatmap plots
- `save_output=True`: Save all results to files

In [ ]:
# Create operation with production-style configuration
operation = CorrelationOperation(
    # Core parameters
    field1=field1,                       # From config
    field2=field2,                       # From config
    method=None,                         # Auto-detect method
    
    # Null handling
    null_handling='drop',                # Drop nulls
    
    # MVF parser (for multi-valued fields)
    mvf_parser=None,                     # No MVF parsing
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,             # Disable for small data
    parallel_processes=1,
    use_dask=False,
    chunk_size=10000,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,         # Create visualization artifacts
    save_output=True,                    # Save results to files
    force_recalculation=False            # Use cache when available
)

print("✓ Operation configured")
print(f"  Field 1:          {operation.field1}")
print(f"  Field 2:          {operation.field2}")
print(f"  Method:           {operation.method or 'auto-detect'}")
print(f"  Null handling:    {operation.null_handling}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing correlation analysis...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and display correlation results
- Review correlation coefficient and significance
- Check null value handling statistics

**What you'll see:**
- Generated artifacts list (correlation JSON, visualizations)
- Field information (names, types, sample size)
- Correlation results (method used, coefficient, interpretation)
- P-value and statistical significance (if p < 0.05)
- Null handling statistics (rows dropped/kept)
- Visual bar representation of correlation strength

**Note:** Correlation coefficient ranges from -1 (perfect negative) to +1 (perfect positive). Values near 0 indicate weak/no correlation.

**Generated artifacts:**
- Correlation JSON in output/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load correlation file
corr_files = list(task_dir.glob('output/*_correlation_*.json'))
if corr_files:
    corr_file = corr_files[0]
    
    with open(corr_file, 'r') as f:
        corr_data = json.load(f)
    
    print("\n" + "=" * 80)
    print("📊 CORRELATION ANALYSIS RESULTS")
    print("=" * 80)
    
    # Display field information
    print("\n🔍 Field Information:")
    print("-" * 60)
    print(f"  Field 1: {corr_data.get('field1', 'N/A')} ({corr_data.get('field1_type', 'unknown')})")
    print(f"  Field 2: {corr_data.get('field2', 'N/A')} ({corr_data.get('field2_type', 'unknown')})")
    print(f"  Sample size: {corr_data.get('sample_size', 0):,}")
    
    # Display correlation results
    print("\n📈 Correlation Results:")
    print("-" * 60)
    method = corr_data.get('method', 'unknown')
    coefficient = corr_data.get('correlation_coefficient', 0)
    print(f"  Method: {method.replace('_', ' ').title()}")
    print(f"  Correlation coefficient: {coefficient:.4f}")
    
    # Display p-value if available
    if 'p_value' in corr_data and corr_data['p_value'] is not None:
        p_value = corr_data['p_value']
        print(f"  P-value: {p_value:.6f}")
        print(f"  Statistically significant (p < 0.05): {'Yes' if p_value < 0.05 else 'No'}")
    
    # Display interpretation
    print(f"\n  Interpretation: {corr_data.get('interpretation', 'N/A')}")
    
    # Display null statistics
    if 'null_stats' in corr_data:
        null_stats = corr_data['null_stats']
        print("\n⚠️  Null Value Statistics:")
        print("-" * 60)
        print(f"  Total rows: {null_stats.get('total_rows', 0):,}")
        print(f"  Rows with nulls: {null_stats.get('null_rows', 0):,}")
        print(f"  Null percentage: {null_stats.get('null_percentage', 0):.2f}%")
        print(f"  Rows after handling: {null_stats.get('rows_after_handling', 0):,}")
    
    # Visual representation of correlation strength
    print("\n📊 Correlation Strength Visualization:")
    print("-" * 60)
    abs_coef = abs(coefficient)
    bar_length = int(abs_coef * 50)
    bar = '█' * bar_length
    direction = '→' if coefficient >= 0 else '←'
    print(f"  {direction} {bar} {abs_coef:.2f}")
    
    # Display result metrics
    if result.metrics:
        print("\n" + "=" * 80)
        print("📊 SUMMARY METRICS")
        print("=" * 80)
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
            else:
                print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Correlation ranges from -1 (perfect negative) to +1 (perfect positive)!")
else:
    print("⚠️  No correlation file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files in directory structure
- Navigate to task_dir for manual inspection
- Verify file counts and sizes

**What you'll see:**
- Directory structure with file counts per folder
- File names with sizes (KB) for up to 5 files per folder
- Full path to task directory for external access
- Total artifacts confirmation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Correlation JSON
├── visualizations/   # PNG charts (scatter/boxplot/heatmap)
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

# List all subdirectories and files
for subdir in ['output', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:  # Show first 5 files
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")
        if len(files) > 5:
            print(f"   ... and {len(files) - 5} more files")
        print()

print("=" * 80)
print("\n✅ All artifacts saved successfully!")
print(f"\n💡 TIP: Navigate to {task_dir.absolute()} to inspect files manually")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance metrics (if available)
2. **Correlation Results**: Detailed correlation analysis
3. **Visualizations**: Scatter plots, boxplots, or heatmaps based on field types

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            # Use only filtered files
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            # Fallback: nothing left after filtering → use all
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # Display metadata
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # Display key metrics
            print("\n📊 KEY METRICS:")
            metric_keys = [
                ('correlation_method', 'Correlation Method'),
                ('correlation_coefficient', 'Correlation Coefficient'),
                ('p_value', 'P-Value'),
                ('statistically_significant', 'Statistically Significant'),
                ('sample_size', 'Sample Size'),
                ('total_rows', 'Total Rows'),
                ('null_rows', 'Null Rows'),
                ('null_percentage', 'Null Percentage'),
                ('interpretation', 'Interpretation')
            ]
            
            for key, label in metric_keys:
                if key in metrics:
                    value = metrics[key]
                    if isinstance(value, float):
                        print(f"   {label}: {value:.4f}")
                    else:
                        print(f"   {label}: {value}")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. CORRELATION RESULTS (NEWEST)
print("\n\n2️⃣ CORRELATION RESULTS (Detailed):")
print("-" * 80)

output_dir = task_dir / "output"

if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")

else:
    corr_files = list(output_dir.glob("*_correlation_*.json"))

    if not corr_files:
        print("⚠️  No correlation files found")

    else:
        # Pick newest
        corr_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_corr_file = corr_files[0]

        mtime = datetime.fromtimestamp(latest_corr_file.stat().st_mtime)
        print(f"✓ Found {len(corr_files)} correlation file(s)")
        print(f"📄 Loading LATEST: {latest_corr_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_corr_file.stat().st_size / 1024:.1f} KB")
        print("=" * 80)

        try:
            with open(latest_corr_file, "r", encoding="utf-8") as f:
                corr_data = json.load(f)

            # FIELD OVERVIEW
            print("\n📊 FIELD OVERVIEW:")
            print("-" * 60)

            field1 = corr_data.get("field1", "N/A")
            field2 = corr_data.get("field2", "N/A")
            field1_type = corr_data.get("field1_type", "unknown")
            field2_type = corr_data.get("field2_type", "unknown")

            print(f"  Field 1 : {field1} ({field1_type})")
            print(f"  Field 2 : {field2} ({field2_type})")

            # ─────────────────────────────────────────────
            # CORRELATION METRICS
            # ─────────────────────────────────────────────
            print("\n📈 CORRELATION METRICS:")
            print("-" * 60)

            method = corr_data.get("method", "unknown")
            coeff = corr_data.get("correlation_coefficient")

            print(f"  Method       : {method.replace('_', ' ').title()}")
            if isinstance(coeff, (int, float)):
                print(f"  Coefficient  : {coeff:.6f}")
            else:
                print(f"  Coefficient  : N/A")

            # STATISTICAL SIGNIFICANCE
            if corr_data.get("p_value") is not None:
                p_val = corr_data["p_value"]

                print("\n🧪 STATISTICAL SIGNIFICANCE:")
                print("-" * 60)
                print(f"  P-value          : {p_val:.6f}")
                print(f"  Significant (α=0.05): {'Yes ✓' if p_val < 0.05 else 'No ✗'}")

                if p_val < 0.01:
                    print(f"  Highly significant (α=0.01): Yes ✓✓")

            # INTERPRETATION
            if "interpretation" in corr_data:
                print("\n🧠 INTERPRETATION:")
                print("-" * 60)
                print(f"  {corr_data['interpretation']}")

            # SAMPLE INFORMATION
            print("\n📦 SAMPLE INFORMATION:")
            print("-" * 60)

            sample_size = corr_data.get("sample_size", 0)
            print(f"  Sample Size : {sample_size:,} records")

            # NULL HANDLING DETAILS
            if "null_stats" in corr_data:
                ns = corr_data["null_stats"]

                print("\n🚫 NULL HANDLING:")
                print("-" * 60)

                print(f"  Total Rows        : {ns.get('total_rows', 0):,}")
                print(f"  Rows With Nulls   : {ns.get('null_rows', 0):,} "
                      f"({ns.get('null_percentage', 0):.2f}%)")
                print(f"  Rows Used         : {ns.get('rows_after_handling', 0):,}")
                print(f"  Rows Removed      : {ns.get('rows_removed', 0):,}")

                if "field_nulls" in ns:
                    print("\n  Nulls Per Field:")
                    for field, count in ns["field_nulls"].items():
                        print(f"    • {field}: {count}")

            # VISUALIZATION METADATA (NO PLOTTING HERE)
            if "plot_data" in corr_data:
                plot_data = corr_data["plot_data"]

                print("\n📊 VISUALIZATION INFO:")
                print("-" * 60)

                print(f"  Plot Type : {plot_data.get('type', 'unknown').upper()}")
                print(f"  X Label   : {plot_data.get('x_label', field1)}")
                print(f"  Y Label   : {plot_data.get('y_label', field2)}")

                x_vals = plot_data.get("x_values", [])
                y_vals = plot_data.get("y_values", [])

                if isinstance(x_vals, list) and isinstance(y_vals, list):
                    print(f"  Sample Points Shown: {min(len(x_vals), len(y_vals))}")

            print("\n✅ Correlation analysis rendering completed successfully.")

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading correlation results: {e}")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').replace('correlation plot', 'Correlation').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV from examples/data_examples/  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with field pairs  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review correlation coefficients and significance  

**Key takeaways:**
- Automatic correlation method selection based on field types
- **Pearson correlation**: For numeric-numeric relationships
- **Cramer's V**: For categorical-categorical relationships
- **Correlation ratio**: For categorical-numeric relationships
- **Point-biserial**: For binary-numeric relationships
- P-values indicate statistical significance (< 0.05 = significant)
- Correlation ranges from -1 (perfect negative) to +1 (perfect positive)
- Null handling strategies: drop, fill, pairwise
- All analysis saved in structured directories

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_correlation_analyzer_advanced.ipynb`** includes:
- **Correlation matrix** for multiple fields at once
- Batch correlation analysis for field pairs
- Advanced visualization options (heatmaps, matrices)
- Multi-valued field (MVF) parsing
- Custom correlation methods
- Performance optimization for large datasets

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Statistical Analysis Guide](../../docs/statistical_analysis.md)
- 📋 [Correlation Methods Reference](../../docs/correlation_methods.md)